In [1]:
from keras.layers import GRU
from keras.layers import Input, Dense
from keras.models import Model
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.optimizers import Adam
import tensorflow as tf
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from project_brain_decoder.config import get_project_root
from project_brain_decoder.io.nwb_loader import load_nwb
from sklearn.metrics import r2_score

tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
batch_size, window_size, input_dim = 128, 30, 192

In [2]:
folder = get_project_root() / "data" / "raw"
files = list(folder.glob("*.nwb"))

In [ ]:
train = files[:187] # 60%
val = files[187:249] # 20%
test = files[249:] # 20%

In [ ]:
np.random.shuffle(files)

In [3]:
def make_windows(neural: np.array, # shape(T, C) - time * channels
                 targets: np.array, # shape(T,) or (T, out_dim)
                 window_size: int,
                 stride: int=10) -> tuple[np.array, np.array]:
    """Slice into (window_size, C) windows; targets aligned to last timestep of each window"""
    T, C = neural.shape
    X = np.lib.stride_tricks.sliding_window_view(neural, window_size, axis=0)[::stride] # (n_windows, C, window_size)
    X = X.transpose(0, 2, 1) # (n_windows, window_size, C)
    # target for each window = value at the end of the window
    y = targets[window_size - 1 :: stride][:X.shape[0]]
    return X, y

In [5]:
neural_scaler = StandardScaler()
targets_scaler = StandardScaler()

In [ ]:
for file in train:
    session = load_nwb(file)
    neural = np.concatenate([session["neural_spiking_band"], session["neural_threshold_crossings"]])
    targets = np.column_stack([session["target_index_velocity"], session["target_mrs_velocity"]])
    neural_scaler.partial_fit(neural)
    targets_scaler.partial_fit(targets)

In [ ]:
def get_gru(window_size, input_dim, GRU):
    input_layer = Input(shape=(window_size, input_dim))
    output_layer = GRU(units=64, dropout=0.5, return_sequences=True)(input_layer)
    output_layer = GRU(units=32, dropout=0.5)(output_layer)
    output_layer = Dense(1)(output_layer)
    gru = Model(inputs=[input_layer], outputs=[output_layer])
    gru.compile(optimizer=Adam(learning_rate=0.0005), loss='mse')
    return gru

In [6]:
if __name__ == "__main__":
    main(model=get_gru(window_size=window_size,
                       input_dim=input_dim,
                       GRU=GRU))

Epoch 1/10
1845/1845 ━━━━━━━━━━━━━━━━━━━━ 57s 30ms/step - loss: 0.9058 - val_loss: 1.1765
Epoch 2/10
 133/1845 ━━━━━━━━━━━━━━━━━━━━ 51s 30ms/step - loss: 0.9901

KeyboardInterrupt: 